In [ ]:
# Mount Drive (Colab)
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# =========================
# mobilenetv3small.py
# =========================

import os
import json
import time
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV3Small
from tensorflow.keras.applications.mobilenet_v3 import preprocess_input
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score, precision_score, recall_score

# =============== CONFIG ===============
SEED = 42
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

HEAD_EPOCHS = 5   # change to 5 later
FINE_TUNE_EPOCHS = 10

LR_HEAD = 1e-3
LR_FINE = 1e-5

#DATA_ROOT = "/content/drive/MyDrive/CV/dataset"
TRAIN_DIR = os.path.join(DATA_ROOT, "train")
VAL_DIR   = os.path.join(DATA_ROOT, "val")
TEST_DIR  = os.path.join(DATA_ROOT, "test")

RESULTS_DIR = "/content/drive/MyDrive/CV/plant_results"
MODELS_DIR = os.path.join(RESULTS_DIR, "saved_models")
PLOTS_DIR  = os.path.join(RESULTS_DIR, "plots")
CSV_DIR    = os.path.join(RESULTS_DIR, "csv")
META_DIR   = os.path.join(RESULTS_DIR, "metadata")

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)
os.makedirs(CSV_DIR, exist_ok=True)
os.makedirs(META_DIR, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

AUTOTUNE = tf.data.AUTOTUNE

MODEL_FAMILY = "MobileNetV3Small"
EXPERIMENTS = [
    {"name": "MobileNetV3Small_baseline", "attention": None},
    {"name": "MobileNetV3Small_SE", "attention": "SE"},
    {"name": "MobileNetV3Small_CBAM", "attention": "CBAM"},
]

# =========================
# DATA
# =========================
class_names = sorted([d for d in os.listdir(TRAIN_DIR) if os.path.isdir(os.path.join(TRAIN_DIR, d))])
num_classes = len(class_names)

with open(os.path.join(META_DIR, "class_names.json"), "w") as f:
    json.dump(class_names, f, indent=2)

def collect_image_paths(root_dir):
    rows = []
    class_to_idx = {c: i for i, c in enumerate(class_names)}
    for cls in class_names:
        cls_dir = os.path.join(root_dir, cls)
        for fname in sorted(os.listdir(cls_dir)):
            fpath = os.path.join(cls_dir, fname)
            if os.path.isfile(fpath):
                rows.append({
                    "image_path": fpath,
                    "class_name": cls,
                    "label_idx": class_to_idx[cls]
                })
    return pd.DataFrame(rows)

train_index_df = collect_image_paths(TRAIN_DIR)
val_index_df   = collect_image_paths(VAL_DIR)
test_index_df  = collect_image_paths(TEST_DIR)

train_index_df.to_csv(os.path.join(CSV_DIR, "train_index.csv"), index=False)
val_index_df.to_csv(os.path.join(CSV_DIR, "val_index.csv"), index=False)
test_index_df.to_csv(os.path.join(CSV_DIR, "test_index.csv"), index=False)

print("Train class counts:")
print(train_index_df["class_name"].value_counts())
print("\nVal class counts:")
print(val_index_df["class_name"].value_counts())
print("\nTest class counts:")
print(test_index_df["class_name"].value_counts())

train_ds_raw = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    labels="inferred",
    label_mode="int",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=SEED
)

val_ds_raw = tf.keras.utils.image_dataset_from_directory(
    VAL_DIR,
    labels="inferred",
    label_mode="int",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_ds_raw = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    labels="inferred",
    label_mode="int",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.08),
    layers.RandomZoom(0.10),
    layers.RandomTranslation(0.05, 0.05),
    layers.RandomContrast(0.08),
], name="augmentation")

def make_dataset(ds_raw, training=False):
    def _map_fn(x, y):
        x = tf.cast(x, tf.float32)
        if training:
            x = data_augmentation(x, training=True)
        x = preprocess_input(x)
        return x, y
    return ds_raw.map(_map_fn, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)

# =========================
# CUSTOM LAYERS FOR CBAM
# =========================
class ChannelAvgPool(layers.Layer):
    def call(self, inputs):
        return tf.reduce_mean(inputs, axis=-1, keepdims=True)

    def get_config(self):
        return super().get_config()

class ChannelMaxPool(layers.Layer):
    def call(self, inputs):
        return tf.reduce_max(inputs, axis=-1, keepdims=True)

    def get_config(self):
        return super().get_config()

# =========================
# ATTENTION BLOCKS
# =========================
def se_block(inputs, ratio=8, name="se"):
    channels = inputs.shape[-1]
    x = layers.GlobalAveragePooling2D(name=f"{name}_gap")(inputs)
    x = layers.Reshape((1, 1, channels), name=f"{name}_reshape")(x)
    x = layers.Dense(max(channels // ratio, 1), activation="relu", name=f"{name}_fc1")(x)
    x = layers.Dense(channels, activation="sigmoid", name=f"{name}_fc2")(x)
    x = layers.Multiply(name=f"{name}_scale")([inputs, x])
    return x

def cbam_block(inputs, ratio=8, name="cbam"):
    channels = inputs.shape[-1]

    # Channel attention
    avg_pool = layers.GlobalAveragePooling2D(name=f"{name}_gap_avg")(inputs)
    max_pool = layers.GlobalMaxPooling2D(name=f"{name}_gap_max")(inputs)

    shared_dense1 = layers.Dense(max(channels // ratio, 1), activation="relu", name=f"{name}_mlp1")
    shared_dense2 = layers.Dense(channels, name=f"{name}_mlp2")

    avg_out = shared_dense2(shared_dense1(avg_pool))
    max_out = shared_dense2(shared_dense1(max_pool))

    ch_attn = layers.Add(name=f"{name}_ch_add")([avg_out, max_out])
    ch_attn = layers.Activation("sigmoid", name=f"{name}_ch_sigmoid")(ch_attn)
    ch_attn = layers.Reshape((1, 1, channels), name=f"{name}_ch_reshape")(ch_attn)

    x = layers.Multiply(name=f"{name}_ch_scale")([inputs, ch_attn])

    # Spatial attention
    avg_pool_sp = ChannelAvgPool(name=f"{name}_sp_avg")(x)
    max_pool_sp = ChannelMaxPool(name=f"{name}_sp_max")(x)
    concat = layers.Concatenate(axis=-1, name=f"{name}_sp_concat")([avg_pool_sp, max_pool_sp])

    sp_attn = layers.Conv2D(
        1,
        kernel_size=7,
        padding="same",
        activation="sigmoid",
        name=f"{name}_sp_conv"
    )(concat)

    out = layers.Multiply(name=f"{name}_sp_scale")([x, sp_attn])

    return out

# =========================
# MODEL
# =========================
def build_model(attention_type=None):
    inputs = keras.Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 3))
    backbone = MobileNetV3Small(include_top=False, weights="imagenet", input_tensor=inputs)
    backbone.trainable = False

    x = backbone.output

    if attention_type == "SE":
        x = se_block(x, name="mobilenetv3_se")
    elif attention_type == "CBAM":
        x = cbam_block(x, name="mobilenetv3_cbam")

    x = layers.GlobalAveragePooling2D(name="gap")(x)
    x = layers.Dropout(0.3, name="dropout")(x)
    outputs = layers.Dense(num_classes, activation="softmax", name="classifier")(x)

    model = keras.Model(inputs, outputs, name=f"{MODEL_FAMILY}_{attention_type or 'baseline'}")
    return model, backbone

# =========================
# UTILITIES
# =========================
def save_history_plot(history_dict, title_prefix, out_path):
    epochs = range(1, len(history_dict["loss"]) + 1)

    plt.figure(figsize=(12, 4))

    plt.subplot(1, 2, 1)
    plt.plot(epochs, history_dict["loss"], label="train_loss")
    plt.plot(epochs, history_dict["val_loss"], label="val_loss")
    plt.title(f"{title_prefix} Loss")
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(epochs, history_dict["accuracy"], label="train_acc")
    plt.plot(epochs, history_dict["val_accuracy"], label="val_acc")
    plt.title(f"{title_prefix} Accuracy")
    plt.legend()

    plt.tight_layout()
    plt.savefig(out_path, dpi=200, bbox_inches="tight")
    plt.close()

def save_confusion_matrix(cm, labels, out_path, title):
    # normalize (row-wise)
    cm = cm.astype('float') / cm.sum(axis=1, keepdims=True)
    cm = np.nan_to_num(cm)  # handle division by zero

    plt.figure(figsize=(8, 6))

    # white → blue colormap
    plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)

    plt.title(title)
    plt.colorbar()

    tick_marks = np.arange(len(labels))
    plt.xticks(tick_marks, labels, rotation=45, ha="right")
    plt.yticks(tick_marks, labels)

    # threshold for text color
    thresh = cm.max() / 2.0 if cm.max() > 0 else 0

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, f"{cm[i, j]:.2f}",
                     horizontalalignment="center",
                     color="white" if cm[i, j] > thresh else "black")

    plt.ylabel("True")
    plt.xlabel("Predicted")
    plt.tight_layout()
    plt.savefig(out_path, dpi=200, bbox_inches="tight")
    plt.close()

def evaluate_model(model, test_ds, exp_name):
    y_true, y_prob = [], []

    for x_batch, y_batch in test_ds:
        prob = model.predict(x_batch, verbose=0)
        y_prob.append(prob)
        y_true.extend(y_batch.numpy())

    y_prob = np.concatenate(y_prob, axis=0)
    y_pred = np.argmax(y_prob, axis=1)
    y_true = np.array(y_true)

    acc = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average="macro", zero_division=0)
    recall = recall_score(y_true, y_pred, average="macro", zero_division=0)
    f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    cm = confusion_matrix(y_true, y_pred)

    report = classification_report(
        y_true, y_pred,
        target_names=class_names,
        output_dict=True,
        zero_division=0
    )

    pred_df = test_index_df.copy()
    pred_df["pred_idx"] = y_pred
    pred_df["pred_class"] = [class_names[i] for i in y_pred]
    pred_df["correct"] = (pred_df["label_idx"] == pred_df["pred_idx"]).astype(int)
    pred_df.to_csv(os.path.join(CSV_DIR, f"{exp_name}_test_predictions.csv"), index=False)

    pd.DataFrame(cm, index=class_names, columns=class_names).to_csv(
        os.path.join(CSV_DIR, f"{exp_name}_confusion_matrix.csv")
    )

    save_confusion_matrix(
        cm,
        class_names,
        os.path.join(PLOTS_DIR, f"{exp_name}_confusion_matrix.png"),
        f"{exp_name} Confusion Matrix"
    )

    with open(os.path.join(CSV_DIR, f"{exp_name}_classification_report.json"), "w") as f:
        json.dump(report, f, indent=2)

    return {
        "Model": exp_name,
        "Accuracy": float(acc),
        "Precision_macro": float(precision),
        "Recall_macro": float(recall),
        "F1_macro": float(f1),
        "Params": int(model.count_params())
    }

def load_saved_model(model_path):
    return keras.models.load_model(
        model_path,
        compile=False,
        safe_mode=False,
        custom_objects={
            "ChannelAvgPool": ChannelAvgPool,
            "ChannelMaxPool": ChannelMaxPool
        }
    )

# =========================
# OPTIONAL: DELETE OLD BROKEN CBAM MODEL
# =========================
old_cbam_path = os.path.join(MODELS_DIR, "MobileNetV3Small_CBAM.keras")
if os.path.exists(old_cbam_path):
    try:
        os.remove(old_cbam_path)
        print("Deleted old broken CBAM model file.")
    except Exception as e:
        print("Could not delete old CBAM model:", e)

# =========================
# TRAIN LOOP
# =========================
train_ds = make_dataset(train_ds_raw, training=True)
val_ds   = make_dataset(val_ds_raw, training=False)
test_ds  = make_dataset(test_ds_raw, training=False)

all_rows = []

for exp in EXPERIMENTS:
    exp_name = exp["name"]
    attention = exp["attention"]

    print(f"\n===== Training {exp_name} =====")

    model, backbone = build_model(attention_type=attention)

    ckpt_path = os.path.join(MODELS_DIR, f"{exp_name}.keras")

    callbacks = [
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, verbose=1),
        keras.callbacks.ModelCheckpoint(ckpt_path, monitor="val_accuracy", save_best_only=True, verbose=1)
    ]

    model.compile(
        optimizer=keras.optimizers.Adam(LR_HEAD),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    hist1 = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=HEAD_EPOCHS,
        callbacks=callbacks,
        verbose=1
    )

    backbone.trainable = True
    total_layers = len(backbone.layers)
    freeze_until = int(total_layers * 0.8)

    for layer in backbone.layers[:freeze_until]:
        layer.trainable = False

    model.compile(
        optimizer=keras.optimizers.Adam(LR_FINE),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    hist2 = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=FINE_TUNE_EPOCHS,
        callbacks=callbacks,
        verbose=1
    )

    full_hist = {}
    for k in hist1.history.keys():
        full_hist[k] = hist1.history[k] + hist2.history[k]

    with open(os.path.join(CSV_DIR, f"{exp_name}_history.json"), "w") as f:
        json.dump(full_hist, f, indent=2)

    save_history_plot(
        full_hist,
        exp_name,
        os.path.join(PLOTS_DIR, f"{exp_name}_history.png")
    )

    best_model = load_saved_model(ckpt_path)
    best_model.compile(
        optimizer=keras.optimizers.Adam(LR_FINE),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    row = evaluate_model(best_model, test_ds, exp_name)
    row["Model_Path"] = ckpt_path
    all_rows.append(row)

summary_df = pd.DataFrame(all_rows).sort_values("F1_macro", ascending=False)
summary_df.to_csv(os.path.join(CSV_DIR, "mobilenetv3small_summary.csv"), index=False)

print("\nSaved:", os.path.join(CSV_DIR, "mobilenetv3small_summary.csv"))
print(summary_df)

Train class counts:
class_name
YellowLeaf_Curl_Virus    2246
Bacterial_spot           1488
Tomato_Late_blight       1336
healthy                  1113
Name: count, dtype: int64

Val class counts:
class_name
YellowLeaf_Curl_Virus    481
Bacterial_spot           319
Tomato_Late_blight       286
healthy                  238
Name: count, dtype: int64

Test class counts:
class_name
YellowLeaf_Curl_Virus    482
Bacterial_spot           320
Tomato_Late_blight       287
healthy                  240
Name: count, dtype: int64
Found 6182 files belonging to 4 classes.
Found 1324 files belonging to 4 classes.
Found 1329 files belonging to 4 classes.

===== Training MobileNetV3Small_baseline =====
4334752/4334752 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/5
194/194 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.6608 - loss: 0.8757
Epoch 1: val_accuracy improved from None to 0.92069, saving model to /content/drive/MyDrive/CV/plant_results/saved_models/MobileNetV3Small_baseline.keras

Epoch 1: finished s

In [ ]:
# =========================
# train_efficientnetb0.py
# =========================

import os
import json
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score, precision_score, recall_score

# =============== CONFIG ===============
SEED = 42
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

HEAD_EPOCHS = 5
FINE_TUNE_EPOCHS = 10

LR_HEAD = 1e-3
LR_FINE = 1e-5

#DATA_ROOT = "/content/drive/MyDrive/CV/dataset"
TRAIN_DIR = os.path.join(DATA_ROOT, "train")
VAL_DIR   = os.path.join(DATA_ROOT, "val")
TEST_DIR  = os.path.join(DATA_ROOT, "test")

RESULTS_DIR = "/content/drive/MyDrive/CV/plant_results"
MODELS_DIR = os.path.join(RESULTS_DIR, "saved_models")
PLOTS_DIR  = os.path.join(RESULTS_DIR, "plots")
CSV_DIR    = os.path.join(RESULTS_DIR, "csv")
META_DIR   = os.path.join(RESULTS_DIR, "metadata")

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)
os.makedirs(CSV_DIR, exist_ok=True)
os.makedirs(META_DIR, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

AUTOTUNE = tf.data.AUTOTUNE

MODEL_FAMILY = "EfficientNetB0"
EXPERIMENTS = [
    {"name": "EfficientNetB0_baseline", "attention": None},
    {"name": "EfficientNetB0_CBAM", "attention": "CBAM"},
]

# =========================
# DATA
# =========================
class_names = sorted([d for d in os.listdir(TRAIN_DIR) if os.path.isdir(os.path.join(TRAIN_DIR, d))])
num_classes = len(class_names)

with open(os.path.join(META_DIR, "class_names.json"), "w") as f:
    json.dump(class_names, f, indent=2)

def collect_image_paths(root_dir):
    rows = []
    class_to_idx = {c: i for i, c in enumerate(class_names)}
    for cls in class_names:
        cls_dir = os.path.join(root_dir, cls)
        for fname in sorted(os.listdir(cls_dir)):
            fpath = os.path.join(cls_dir, fname)
            if os.path.isfile(fpath):
                rows.append({
                    "image_path": fpath,
                    "class_name": cls,
                    "label_idx": class_to_idx[cls]
                })
    return pd.DataFrame(rows)

train_index_df = collect_image_paths(TRAIN_DIR)
val_index_df   = collect_image_paths(VAL_DIR)
test_index_df  = collect_image_paths(TEST_DIR)

train_index_df.to_csv(os.path.join(CSV_DIR, "train_index.csv"), index=False)
val_index_df.to_csv(os.path.join(CSV_DIR, "val_index.csv"), index=False)
test_index_df.to_csv(os.path.join(CSV_DIR, "test_index.csv"), index=False)

print("Train class counts:")
print(train_index_df["class_name"].value_counts())
print("\nVal class counts:")
print(val_index_df["class_name"].value_counts())
print("\nTest class counts:")
print(test_index_df["class_name"].value_counts())

train_ds_raw = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    labels="inferred",
    label_mode="int",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=SEED
)

val_ds_raw = tf.keras.utils.image_dataset_from_directory(
    VAL_DIR,
    labels="inferred",
    label_mode="int",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_ds_raw = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    labels="inferred",
    label_mode="int",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.08),
    layers.RandomZoom(0.10),
    layers.RandomTranslation(0.05, 0.05),
    layers.RandomContrast(0.08),
], name="augmentation")

def make_dataset(ds_raw, training=False):
    def _map_fn(x, y):
        x = tf.cast(x, tf.float32)
        if training:
            x = data_augmentation(x, training=True)
        x = preprocess_input(x)
        return x, y
    return ds_raw.map(_map_fn, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)

# =========================
# CUSTOM LAYERS FOR CBAM
# =========================
class ChannelAvgPool(layers.Layer):
    def call(self, inputs):
        return tf.reduce_mean(inputs, axis=-1, keepdims=True)

    def get_config(self):
        return super().get_config()

class ChannelMaxPool(layers.Layer):
    def call(self, inputs):
        return tf.reduce_max(inputs, axis=-1, keepdims=True)

    def get_config(self):
        return super().get_config()

# =========================
# ATTENTION BLOCKS
# =========================
def cbam_block(inputs, ratio=8, name="cbam"):
    channels = inputs.shape[-1]

    # Channel attention
    avg_pool = layers.GlobalAveragePooling2D(name=f"{name}_gap_avg")(inputs)
    max_pool = layers.GlobalMaxPooling2D(name=f"{name}_gap_max")(inputs)

    shared_dense1 = layers.Dense(max(channels // ratio, 1), activation="relu", name=f"{name}_mlp1")
    shared_dense2 = layers.Dense(channels, name=f"{name}_mlp2")

    avg_out = shared_dense2(shared_dense1(avg_pool))
    max_out = shared_dense2(shared_dense1(max_pool))

    ch_attn = layers.Add(name=f"{name}_ch_add")([avg_out, max_out])
    ch_attn = layers.Activation("sigmoid", name=f"{name}_ch_sigmoid")(ch_attn)
    ch_attn = layers.Reshape((1, 1, channels), name=f"{name}_ch_reshape")(ch_attn)

    x = layers.Multiply(name=f"{name}_ch_scale")([inputs, ch_attn])

    # Spatial attention
    avg_pool_sp = ChannelAvgPool(name=f"{name}_sp_avg")(x)
    max_pool_sp = ChannelMaxPool(name=f"{name}_sp_max")(x)
    concat = layers.Concatenate(axis=-1, name=f"{name}_sp_concat")([avg_pool_sp, max_pool_sp])

    sp_attn = layers.Conv2D(
        1,
        kernel_size=7,
        padding="same",
        activation="sigmoid",
        name=f"{name}_sp_conv"
    )(concat)

    out = layers.Multiply(name=f"{name}_sp_scale")([x, sp_attn])

    return out

# =========================
# MODEL
# =========================
def build_model(attention_type=None):
    inputs = keras.Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 3))
    backbone = EfficientNetB0(include_top=False, weights="imagenet", input_tensor=inputs)
    backbone.trainable = False

    x = backbone.output

    if attention_type == "CBAM":
        x = cbam_block(x, name="efficientnetb0_cbam")

    x = layers.GlobalAveragePooling2D(name="gap")(x)
    x = layers.Dropout(0.3, name="dropout")(x)
    outputs = layers.Dense(num_classes, activation="softmax", name="classifier")(x)

    model = keras.Model(inputs, outputs, name=f"{MODEL_FAMILY}_{attention_type or 'baseline'}")
    return model, backbone

# =========================
# UTILITIES
# =========================
def save_history_plot(history_dict, title_prefix, out_path):
    epochs = range(1, len(history_dict["loss"]) + 1)

    plt.figure(figsize=(12, 4))

    plt.subplot(1, 2, 1)
    plt.plot(epochs, history_dict["loss"], label="train_loss")
    plt.plot(epochs, history_dict["val_loss"], label="val_loss")
    plt.title(f"{title_prefix} Loss")
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(epochs, history_dict["accuracy"], label="train_acc")
    plt.plot(epochs, history_dict["val_accuracy"], label="val_acc")
    plt.title(f"{title_prefix} Accuracy")
    plt.legend()

    plt.tight_layout()
    plt.savefig(out_path, dpi=200, bbox_inches="tight")
    plt.close()

def save_confusion_matrix(cm, labels, out_path, title):
    # normalize (row-wise)
    cm = cm.astype('float') / cm.sum(axis=1, keepdims=True)
    cm = np.nan_to_num(cm)  # handle division by zero

    plt.figure(figsize=(8, 6))

    # white → blue colormap
    plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)

    plt.title(title)
    plt.colorbar()

    tick_marks = np.arange(len(labels))
    plt.xticks(tick_marks, labels, rotation=45, ha="right")
    plt.yticks(tick_marks, labels)

    # threshold for text color
    thresh = cm.max() / 2.0 if cm.max() > 0 else 0

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, f"{cm[i, j]:.2f}",
                     horizontalalignment="center",
                     color="white" if cm[i, j] > thresh else "black")

    plt.ylabel("True")
    plt.xlabel("Predicted")
    plt.tight_layout()
    plt.savefig(out_path, dpi=200, bbox_inches="tight")
    plt.close()

def evaluate_model(model, test_ds, exp_name):
    y_true, y_prob = [], []

    for x_batch, y_batch in test_ds:
        prob = model.predict(x_batch, verbose=0)
        y_prob.append(prob)
        y_true.extend(y_batch.numpy())

    y_prob = np.concatenate(y_prob, axis=0)
    y_pred = np.argmax(y_prob, axis=1)
    y_true = np.array(y_true)

    acc = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average="macro", zero_division=0)
    recall = recall_score(y_true, y_pred, average="macro", zero_division=0)
    f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    cm = confusion_matrix(y_true, y_pred)

    report = classification_report(
        y_true,
        y_pred,
        target_names=class_names,
        output_dict=True,
        zero_division=0
    )

    pred_df = test_index_df.copy()
    pred_df["pred_idx"] = y_pred
    pred_df["pred_class"] = [class_names[i] for i in y_pred]
    pred_df["correct"] = (pred_df["label_idx"] == pred_df["pred_idx"]).astype(int)
    pred_df.to_csv(os.path.join(CSV_DIR, f"{exp_name}_test_predictions.csv"), index=False)

    pd.DataFrame(cm, index=class_names, columns=class_names).to_csv(
        os.path.join(CSV_DIR, f"{exp_name}_confusion_matrix.csv")
    )

    save_confusion_matrix(
        cm,
        class_names,
        os.path.join(PLOTS_DIR, f"{exp_name}_confusion_matrix.png"),
        f"{exp_name} Confusion Matrix"
    )

    with open(os.path.join(CSV_DIR, f"{exp_name}_classification_report.json"), "w") as f:
        json.dump(report, f, indent=2)

    return {
        "Model": exp_name,
        "Accuracy": float(acc),
        "Precision_macro": float(precision),
        "Recall_macro": float(recall),
        "F1_macro": float(f1),
        "Params": int(model.count_params())
    }

def load_saved_model(model_path):
    return keras.models.load_model(
        model_path,
        compile=False,
        safe_mode=False,
        custom_objects={
            "ChannelAvgPool": ChannelAvgPool,
            "ChannelMaxPool": ChannelMaxPool
        }
    )

# =========================
# OPTIONAL: DELETE OLD BROKEN CBAM MODEL
# =========================
old_cbam_path = os.path.join(MODELS_DIR, "EfficientNetB0_CBAM.keras")
if os.path.exists(old_cbam_path):
    try:
        os.remove(old_cbam_path)
        print("Deleted old broken EfficientNetB0 CBAM model file.")
    except Exception as e:
        print("Could not delete old EfficientNetB0 CBAM model:", e)

# =========================
# TRAIN LOOP
# =========================
train_ds = make_dataset(train_ds_raw, training=True)
val_ds   = make_dataset(val_ds_raw, training=False)
test_ds  = make_dataset(test_ds_raw, training=False)

all_rows = []

for exp in EXPERIMENTS:
    exp_name = exp["name"]
    attention = exp["attention"]

    print(f"\n===== Training {exp_name} =====")

    model, backbone = build_model(attention_type=attention)
    ckpt_path = os.path.join(MODELS_DIR, f"{exp_name}.keras")

    callbacks = [
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, verbose=1),
        keras.callbacks.ModelCheckpoint(ckpt_path, monitor="val_accuracy", save_best_only=True, verbose=1)
    ]

    model.compile(
        optimizer=keras.optimizers.Adam(LR_HEAD),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    hist1 = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=HEAD_EPOCHS,
        callbacks=callbacks,
        verbose=1
    )

    backbone.trainable = True
    total_layers = len(backbone.layers)
    freeze_until = int(total_layers * 0.8)

    for layer in backbone.layers[:freeze_until]:
        layer.trainable = False

    model.compile(
        optimizer=keras.optimizers.Adam(LR_FINE),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    hist2 = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=FINE_TUNE_EPOCHS,
        callbacks=callbacks,
        verbose=1
    )

    full_hist = {}
    for k in hist1.history.keys():
        full_hist[k] = hist1.history[k] + hist2.history[k]

    with open(os.path.join(CSV_DIR, f"{exp_name}_history.json"), "w") as f:
        json.dump(full_hist, f, indent=2)

    save_history_plot(
        full_hist,
        exp_name,
        os.path.join(PLOTS_DIR, f"{exp_name}_history.png")
    )

    best_model = load_saved_model(ckpt_path)
    best_model.compile(
        optimizer=keras.optimizers.Adam(LR_FINE),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    row = evaluate_model(best_model, test_ds, exp_name)
    row["Model_Path"] = ckpt_path
    all_rows.append(row)

summary_df = pd.DataFrame(all_rows).sort_values("F1_macro", ascending=False)
summary_df.to_csv(os.path.join(CSV_DIR, "efficientnetb0_summary.csv"), index=False)

print("\nSaved:", os.path.join(CSV_DIR, "efficientnetb0_summary.csv"))
print(summary_df)

Train class counts:
class_name
YellowLeaf_Curl_Virus    2246
Bacterial_spot           1488
Tomato_Late_blight       1336
healthy                  1113
Name: count, dtype: int64

Val class counts:
class_name
YellowLeaf_Curl_Virus    481
Bacterial_spot           319
Tomato_Late_blight       286
healthy                  238
Name: count, dtype: int64

Test class counts:
class_name
YellowLeaf_Curl_Virus    482
Bacterial_spot           320
Tomato_Late_blight       287
healthy                  240
Name: count, dtype: int64
Found 6182 files belonging to 4 classes.
Found 1324 files belonging to 4 classes.
Found 1329 files belonging to 4 classes.

===== Training EfficientNetB0_baseline =====
16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/5
194/194 ━━━━━━━━━━━━━━━━━━━━ 0s 604ms/step - accuracy: 0.7390 - loss: 0.7277
Epoch 1: val_accuracy improved from None to 0.93656, saving model to /content/drive/MyDrive/CV/plant_results/saved_models/EfficientNetB0_baseline.keras

Epoch 1: finished 

In [ ]:
# =========================
# train_resnet50.py
# =========================

import os
import json
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score, precision_score, recall_score

# =============== CONFIG ===============
SEED = 42
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

HEAD_EPOCHS = 5
FINE_TUNE_EPOCHS = 10

LR_HEAD = 1e-3
LR_FINE = 1e-5

#DATA_ROOT = "/content/drive/MyDrive/CV/dataset"
TRAIN_DIR = os.path.join(DATA_ROOT, "train")
VAL_DIR   = os.path.join(DATA_ROOT, "val")
TEST_DIR  = os.path.join(DATA_ROOT, "test")

RESULTS_DIR = "/content/drive/MyDrive/CV/plant_results"
MODELS_DIR = os.path.join(RESULTS_DIR, "saved_models")
PLOTS_DIR  = os.path.join(RESULTS_DIR, "plots")
CSV_DIR    = os.path.join(RESULTS_DIR, "csv")
META_DIR   = os.path.join(RESULTS_DIR, "metadata")

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)
os.makedirs(CSV_DIR, exist_ok=True)
os.makedirs(META_DIR, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

AUTOTUNE = tf.data.AUTOTUNE

MODEL_FAMILY = "ResNet50"
EXPERIMENTS = [
    {"name": "ResNet50_baseline", "attention": None},
]

# =========================
# DATA
# =========================
class_names = sorted([d for d in os.listdir(TRAIN_DIR) if os.path.isdir(os.path.join(TRAIN_DIR, d))])
num_classes = len(class_names)

with open(os.path.join(META_DIR, "class_names.json"), "w") as f:
    json.dump(class_names, f, indent=2)

def collect_image_paths(root_dir):
    rows = []
    class_to_idx = {c: i for i, c in enumerate(class_names)}
    for cls in class_names:
        cls_dir = os.path.join(root_dir, cls)
        for fname in sorted(os.listdir(cls_dir)):
            fpath = os.path.join(cls_dir, fname)
            if os.path.isfile(fpath):
                rows.append({
                    "image_path": fpath,
                    "class_name": cls,
                    "label_idx": class_to_idx[cls]
                })
    return pd.DataFrame(rows)

train_index_df = collect_image_paths(TRAIN_DIR)
val_index_df   = collect_image_paths(VAL_DIR)
test_index_df  = collect_image_paths(TEST_DIR)

train_index_df.to_csv(os.path.join(CSV_DIR, "train_index.csv"), index=False)
val_index_df.to_csv(os.path.join(CSV_DIR, "val_index.csv"), index=False)
test_index_df.to_csv(os.path.join(CSV_DIR, "test_index.csv"), index=False)

print("Train class counts:")
print(train_index_df["class_name"].value_counts())
print("\nVal class counts:")
print(val_index_df["class_name"].value_counts())
print("\nTest class counts:")
print(test_index_df["class_name"].value_counts())

train_ds_raw = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    labels="inferred",
    label_mode="int",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=SEED
)

val_ds_raw = tf.keras.utils.image_dataset_from_directory(
    VAL_DIR,
    labels="inferred",
    label_mode="int",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_ds_raw = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    labels="inferred",
    label_mode="int",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.08),
    layers.RandomZoom(0.10),
    layers.RandomTranslation(0.05, 0.05),
    layers.RandomContrast(0.08),
], name="augmentation")

def make_dataset(ds_raw, training=False):
    def _map_fn(x, y):
        x = tf.cast(x, tf.float32)
        if training:
            x = data_augmentation(x, training=True)
        x = preprocess_input(x)
        return x, y
    return ds_raw.map(_map_fn, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)

# =========================
# MODEL
# =========================
def build_model(attention_type=None):
    inputs = keras.Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 3))
    backbone = ResNet50(include_top=False, weights="imagenet", input_tensor=inputs)
    backbone.trainable = False

    x = backbone.output
    x = layers.GlobalAveragePooling2D(name="gap")(x)
    x = layers.Dropout(0.3, name="dropout")(x)
    outputs = layers.Dense(num_classes, activation="softmax", name="classifier")(x)

    model = keras.Model(inputs, outputs, name=f"{MODEL_FAMILY}_{attention_type or 'baseline'}")
    return model, backbone

# =========================
# UTILITIES
# =========================
def save_history_plot(history_dict, title_prefix, out_path):
    epochs = range(1, len(history_dict["loss"]) + 1)

    plt.figure(figsize=(12, 4))

    plt.subplot(1, 2, 1)
    plt.plot(epochs, history_dict["loss"], label="train_loss")
    plt.plot(epochs, history_dict["val_loss"], label="val_loss")
    plt.title(f"{title_prefix} Loss")
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(epochs, history_dict["accuracy"], label="train_acc")
    plt.plot(epochs, history_dict["val_accuracy"], label="val_acc")
    plt.title(f"{title_prefix} Accuracy")
    plt.legend()

    plt.tight_layout()
    plt.savefig(out_path, dpi=200, bbox_inches="tight")
    plt.close()

def save_confusion_matrix(cm, labels, out_path, title):
    # normalize (row-wise)
    cm = cm.astype('float') / cm.sum(axis=1, keepdims=True)
    cm = np.nan_to_num(cm)  # handle division by zero

    plt.figure(figsize=(8, 6))

    # white → blue colormap
    plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)

    plt.title(title)
    plt.colorbar()

    tick_marks = np.arange(len(labels))
    plt.xticks(tick_marks, labels, rotation=45, ha="right")
    plt.yticks(tick_marks, labels)

    # threshold for text color
    thresh = cm.max() / 2.0 if cm.max() > 0 else 0

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, f"{cm[i, j]:.2f}",
                     horizontalalignment="center",
                     color="white" if cm[i, j] > thresh else "black")

    plt.ylabel("True")
    plt.xlabel("Predicted")
    plt.tight_layout()
    plt.savefig(out_path, dpi=200, bbox_inches="tight")
    plt.close()

def evaluate_model(model, test_ds, exp_name):
    y_true, y_prob = [], []

    for x_batch, y_batch in test_ds:
        prob = model.predict(x_batch, verbose=0)
        y_prob.append(prob)
        y_true.extend(y_batch.numpy())

    y_prob = np.concatenate(y_prob, axis=0)
    y_pred = np.argmax(y_prob, axis=1)
    y_true = np.array(y_true)

    acc = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average="macro", zero_division=0)
    recall = recall_score(y_true, y_pred, average="macro", zero_division=0)
    f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    cm = confusion_matrix(y_true, y_pred)

    report = classification_report(
        y_true,
        y_pred,
        target_names=class_names,
        output_dict=True,
        zero_division=0
    )

    pred_df = test_index_df.copy()
    pred_df["pred_idx"] = y_pred
    pred_df["pred_class"] = [class_names[i] for i in y_pred]
    pred_df["correct"] = (pred_df["label_idx"] == pred_df["pred_idx"]).astype(int)
    pred_df.to_csv(os.path.join(CSV_DIR, f"{exp_name}_test_predictions.csv"), index=False)

    pd.DataFrame(cm, index=class_names, columns=class_names).to_csv(
        os.path.join(CSV_DIR, f"{exp_name}_confusion_matrix.csv")
    )

    save_confusion_matrix(
        cm,
        class_names,
        os.path.join(PLOTS_DIR, f"{exp_name}_confusion_matrix.png"),
        f"{exp_name} Confusion Matrix"
    )

    with open(os.path.join(CSV_DIR, f"{exp_name}_classification_report.json"), "w") as f:
        json.dump(report, f, indent=2)

    return {
        "Model": exp_name,
        "Accuracy": float(acc),
        "Precision_macro": float(precision),
        "Recall_macro": float(recall),
        "F1_macro": float(f1),
        "Params": int(model.count_params())
    }

# =========================
# TRAIN LOOP
# =========================
train_ds = make_dataset(train_ds_raw, training=True)
val_ds   = make_dataset(val_ds_raw, training=False)
test_ds  = make_dataset(test_ds_raw, training=False)

all_rows = []

for exp in EXPERIMENTS:
    exp_name = exp["name"]
    attention = exp["attention"]

    print(f"\n===== Training {exp_name} =====")

    model, backbone = build_model(attention_type=attention)
    ckpt_path = os.path.join(MODELS_DIR, f"{exp_name}.keras")

    callbacks = [
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, verbose=1),
        keras.callbacks.ModelCheckpoint(ckpt_path, monitor="val_accuracy", save_best_only=True, verbose=1)
    ]

    model.compile(
        optimizer=keras.optimizers.Adam(LR_HEAD),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    hist1 = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=HEAD_EPOCHS,
        callbacks=callbacks,
        verbose=1
    )

    backbone.trainable = True
    total_layers = len(backbone.layers)
    freeze_until = int(total_layers * 0.8)

    for layer in backbone.layers[:freeze_until]:
        layer.trainable = False

    model.compile(
        optimizer=keras.optimizers.Adam(LR_FINE),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    hist2 = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=FINE_TUNE_EPOCHS,
        callbacks=callbacks,
        verbose=1
    )

    full_hist = {}
    for k in hist1.history.keys():
        full_hist[k] = hist1.history[k] + hist2.history[k]

    with open(os.path.join(CSV_DIR, f"{exp_name}_history.json"), "w") as f:
        json.dump(full_hist, f, indent=2)

    save_history_plot(
        full_hist,
        exp_name,
        os.path.join(PLOTS_DIR, f"{exp_name}_history.png")
    )

    best_model = keras.models.load_model(ckpt_path, compile=False)
    best_model.compile(
        optimizer=keras.optimizers.Adam(LR_FINE),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    row = evaluate_model(best_model, test_ds, exp_name)
    row["Model_Path"] = ckpt_path
    all_rows.append(row)

summary_df = pd.DataFrame(all_rows).sort_values("F1_macro", ascending=False)
summary_df.to_csv(os.path.join(CSV_DIR, "resnet50_summary.csv"), index=False)

print("\nSaved:", os.path.join(CSV_DIR, "resnet50_summary.csv"))
print(summary_df)

Train class counts:
class_name
YellowLeaf_Curl_Virus    2246
Bacterial_spot           1488
Tomato_Late_blight       1336
healthy                  1113
Name: count, dtype: int64

Val class counts:
class_name
YellowLeaf_Curl_Virus    481
Bacterial_spot           319
Tomato_Late_blight       286
healthy                  238
Name: count, dtype: int64

Test class counts:
class_name
YellowLeaf_Curl_Virus    482
Bacterial_spot           320
Tomato_Late_blight       287
healthy                  240
Name: count, dtype: int64
Found 6182 files belonging to 4 classes.
Found 1324 files belonging to 4 classes.
Found 1329 files belonging to 4 classes.

===== Training ResNet50_baseline =====
94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/5
194/194 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.7851 - loss: 0.5599
Epoch 1: val_accuracy improved from None to 0.96073, saving model to /content/drive/MyDrive/CV/plant_results/saved_models/ResNet50_baseline.keras

Epoch 1: finished saving model to

In [2]:
# =========================
# run_final_analysis_t4_optimized.py
# =========================

from google.colab import drive
drive.mount('/content/drive')

# =========================
# IMPORTS
# =========================
import os
import json
import time
import random
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras.applications.mobilenet_v3 import preprocess_input as mobilenet_preprocess
from tensorflow.keras.applications.efficientnet import preprocess_input as efficientnet_preprocess
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet_preprocess
from sklearn.metrics import accuracy_score, f1_score

# =========================
# GPU SETUP
# =========================
print("TensorFlow version:", tf.__version__)
print("Available GPUs:", tf.config.list_physical_devices("GPU"))

gpus = tf.config.list_physical_devices("GPU")
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("GPU memory growth enabled.")
    except RuntimeError as e:
        print("Memory growth setup warning:", e)

print("\n=== NVIDIA GPU INFO ===")
!nvidia-smi

# =========================
# CONFIG
# =========================
SEED = 42
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

DATA_ROOT = "/content/drive/MyDrive/CV/dataset"
TRAIN_DIR = os.path.join(DATA_ROOT, "train")
VAL_DIR   = os.path.join(DATA_ROOT, "val")
TEST_DIR  = os.path.join(DATA_ROOT, "test")

RESULTS_DIR = "/content/drive/MyDrive/CV/plant_results"
MODELS_DIR = os.path.join(RESULTS_DIR, "saved_models")
PLOTS_DIR  = os.path.join(RESULTS_DIR, "plots")
CSV_DIR    = os.path.join(RESULTS_DIR, "csv")
META_DIR   = os.path.join(RESULTS_DIR, "metadata")

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)
os.makedirs(CSV_DIR, exist_ok=True)
os.makedirs(META_DIR, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
AUTOTUNE = tf.data.AUTOTUNE

# =========================
# CUSTOM LAYERS FOR CBAM MODELS
# =========================
class ChannelAvgPool(keras.layers.Layer):
    def call(self, inputs):
        return tf.reduce_mean(inputs, axis=-1, keepdims=True)

    def get_config(self):
        return super().get_config()


class ChannelMaxPool(keras.layers.Layer):
    def call(self, inputs):
        return tf.reduce_max(inputs, axis=-1, keepdims=True)

    def get_config(self):
        return super().get_config()


def load_saved_model(model_path):
    return keras.models.load_model(
        model_path,
        compile=False,
        safe_mode=False,
        custom_objects={
            "ChannelAvgPool": ChannelAvgPool,
            "ChannelMaxPool": ChannelMaxPool
        }
    )

# =========================
# PREPROCESS HELPERS
# =========================
def get_preprocess_fn(model_name):
    if model_name.startswith("MobileNetV3Small"):
        return mobilenet_preprocess
    elif model_name.startswith("EfficientNetB0"):
        return efficientnet_preprocess
    elif model_name.startswith("ResNet50"):
        return resnet_preprocess
    else:
        raise ValueError(f"Unknown model family in {model_name}")


def load_raw_image_array(img_path, target_size=IMG_SIZE):
    img = keras.utils.load_img(img_path, target_size=target_size)
    arr = keras.utils.img_to_array(img).astype(np.float32)
    return arr


def preprocess_single_image(img_path, model_name):
    arr = load_raw_image_array(img_path, target_size=IMG_SIZE)
    arr = np.expand_dims(arr, axis=0)
    arr = get_preprocess_fn(model_name)(arr)
    return arr


def preprocess_batch_image_paths(image_paths, model_name, batch_size=BATCH_SIZE):
    preprocess_fn = get_preprocess_fn(model_name)

    def gen():
        for p in image_paths:
            arr = load_raw_image_array(p, target_size=IMG_SIZE)
            yield arr

    ds = tf.data.Dataset.from_generator(
        gen,
        output_signature=tf.TensorSpec(shape=(IMG_SIZE[0], IMG_SIZE[1], 3), dtype=tf.float32)
    )

    ds = ds.batch(batch_size).map(lambda x: preprocess_fn(x), num_parallel_calls=AUTOTUNE)
    ds = ds.prefetch(AUTOTUNE)
    return ds


def batched_predict_from_paths(model, model_name, image_paths, batch_size=BATCH_SIZE):
    ds = preprocess_batch_image_paths(image_paths, model_name, batch_size=batch_size)
    probs = model.predict(ds, verbose=0)
    pred_idx = np.argmax(probs, axis=1)
    return pred_idx, probs

# =========================
# LOAD METADATA
# =========================
class_names_path = os.path.join(META_DIR, "class_names.json")
test_index_path = os.path.join(CSV_DIR, "test_index.csv")

if not os.path.exists(class_names_path):
    raise FileNotFoundError(f"Missing: {class_names_path}")
if not os.path.exists(test_index_path):
    raise FileNotFoundError(f"Missing: {test_index_path}")

with open(class_names_path, "r") as f:
    class_names = json.load(f)

test_index_df = pd.read_csv(test_index_path)

# =========================
# LOAD SUMMARY CSVs
# =========================
summary_files = [
    os.path.join(CSV_DIR, "mobilenetv3small_summary.csv"),
    os.path.join(CSV_DIR, "efficientnetb0_summary.csv"),
    os.path.join(CSV_DIR, "resnet50_summary.csv"),
]

summary_parts = []
for fpath in summary_files:
    if os.path.exists(fpath):
        summary_parts.append(pd.read_csv(fpath))

if len(summary_parts) == 0:
    raise FileNotFoundError("No model summary CSV found. Run training scripts first.")

summary_df = pd.concat(summary_parts, ignore_index=True)
summary_df = summary_df.sort_values("F1_macro", ascending=False).reset_index(drop=True)
summary_df.to_csv(os.path.join(CSV_DIR, "all_models_summary.csv"), index=False)

print("\n=== Combined Summary ===")
print(summary_df)

# =========================
# EXPERIMENT GROUP 1 + 2
# =========================
baseline_models = ["MobileNetV3Small_baseline", "EfficientNetB0_baseline", "ResNet50_baseline"]
group1_df = summary_df[summary_df["Model"].isin(baseline_models)].copy()
group1_df.to_csv(os.path.join(CSV_DIR, "experiment_group1_baselines.csv"), index=False)

attention_models = ["MobileNetV3Small_SE", "MobileNetV3Small_CBAM", "EfficientNetB0_CBAM"]
group2_df = summary_df[summary_df["Model"].isin(attention_models)].copy()
group2_df.to_csv(os.path.join(CSV_DIR, "experiment_group2_attention.csv"), index=False)

# =========================
# EXPERIMENT GROUP 3
# SEVERITY-AWARE ANALYSIS
# =========================
def estimate_severity_from_image(img_bgr):
    img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)

    lower_green = np.array([25, 20, 20])
    upper_green = np.array([95, 255, 255])
    green_mask = cv2.inRange(hsv, lower_green, upper_green)

    lower_yellow = np.array([10, 30, 30])
    upper_yellow = np.array([35, 255, 255])
    yellow_mask = cv2.inRange(hsv, lower_yellow, upper_yellow)

    lower_brown = np.array([5, 20, 20])
    upper_brown = np.array([25, 255, 180])
    brown_mask = cv2.inRange(hsv, lower_brown, upper_brown)

    disease_mask = cv2.bitwise_or(yellow_mask, brown_mask)

    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    _, leaf_mask2 = cv2.threshold(gray, 20, 255, cv2.THRESH_BINARY)
    leaf_mask = cv2.bitwise_or(green_mask, leaf_mask2)

    leaf_area = np.sum(leaf_mask > 0)
    disease_area = np.sum((disease_mask > 0) & (leaf_mask > 0))
    lesion_ratio = 0.0 if leaf_area == 0 else disease_area / leaf_area

    if lesion_ratio < 0.10:
        severity = "mild"
    elif lesion_ratio < 0.25:
        severity = "moderate"
    else:
        severity = "severe"

    return lesion_ratio, severity


severity_rows = []
for _, row in test_index_df.iterrows():
    img_path = row["image_path"]
    cls_name = row["class_name"]
    img_bgr = cv2.imread(img_path)

    if img_bgr is None:
        continue

    if cls_name.lower() == "healthy":
        lesion_ratio = 0.0
        severity = "healthy"
    else:
        lesion_ratio, severity = estimate_severity_from_image(img_bgr)

    severity_rows.append({
        "image_path": img_path,
        "true_class": cls_name,
        "true_idx": int(row["label_idx"]),
        "lesion_ratio": lesion_ratio,
        "severity": severity
    })

severity_df = pd.DataFrame(severity_rows)
severity_proxy_path = os.path.join(CSV_DIR, "test_severity_proxy.csv")
severity_df.to_csv(severity_proxy_path, index=False)

# Use best model from combined summary
best_model_name = summary_df.iloc[0]["Model"]
best_model_path = summary_df.iloc[0]["Model_Path"]

print("\nBest model for severity analysis:", best_model_name)
best_model = load_saved_model(best_model_path)

preds, probs = batched_predict_from_paths(
    best_model,
    best_model_name,
    severity_df["image_path"].tolist(),
    batch_size=BATCH_SIZE
)

severity_df["pred_idx"] = preds
severity_df["pred_class"] = [class_names[i] for i in preds]
severity_df["correct"] = (severity_df["true_idx"] == severity_df["pred_idx"]).astype(int)
severity_df.to_csv(os.path.join(CSV_DIR, f"{best_model_name}_severity_predictions.csv"), index=False)

sev_rows = []
for sev in ["mild", "moderate", "severe"]:
    sub = severity_df[severity_df["severity"] == sev].copy()
    if len(sub) == 0:
        continue

    y_true = sub["true_idx"].values
    y_pred = sub["pred_idx"].values

    sev_rows.append({
        "Severity": sev,
        "Count": len(sub),
        "Accuracy": float(accuracy_score(y_true, y_pred)),
        "Macro_F1": float(f1_score(y_true, y_pred, average="macro", zero_division=0))
    })

severity_result_df = pd.DataFrame(sev_rows)
severity_result_df.to_csv(os.path.join(CSV_DIR, "experiment_group3_severity_results.csv"), index=False)

print("\n=== Severity Results ===")
print(severity_result_df)

# =========================
# EXPERIMENT GROUP 4
# GRAD-CAM FOR ALL MODELS
# =========================
def find_last_conv_layer(model):
    for layer in reversed(model.layers):
        if isinstance(layer, tf.keras.layers.Conv2D):
            return layer.name
    raise ValueError("No Conv2D layer found.")


def make_gradcam_heatmap(img_array, model, last_conv_layer_name, pred_index=None):
    grad_model = tf.keras.models.Model(
        [model.inputs],
        [model.get_layer(last_conv_layer_name).output, model.output]
    )

    with tf.GradientTape() as tape:
        conv_outputs, preds = grad_model(img_array, training=False)
        if pred_index is None:
            pred_index = tf.argmax(preds[0])
        class_channel = preds[:, pred_index]

    grads = tape.gradient(class_channel, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()


def save_gradcam_overlay(img_path, model, model_name, out_path,
                         true_class=None, severity=None, alpha=0.4):
    img = keras.utils.load_img(img_path, target_size=IMG_SIZE)
    img_array = keras.utils.img_to_array(img)

    x = np.expand_dims(img_array.copy(), axis=0).astype(np.float32)
    x = get_preprocess_fn(model_name)(x)

    last_conv_layer_name = find_last_conv_layer(model)
    heatmap = make_gradcam_heatmap(x, model, last_conv_layer_name)

    img_uint8 = np.uint8(img_array)
    heatmap_resized = cv2.resize(heatmap, (img_uint8.shape[1], img_uint8.shape[0]))
    heatmap_uint8 = np.uint8(255 * heatmap_resized)
    heatmap_color = cv2.applyColorMap(heatmap_uint8, cv2.COLORMAP_JET)

    base_bgr = cv2.cvtColor(img_uint8, cv2.COLOR_RGB2BGR)
    superimposed = cv2.addWeighted(base_bgr, 1 - alpha, heatmap_color, alpha, 0)

    pred = model.predict(x, verbose=0)[0]
    pred_idx = int(np.argmax(pred))
    pred_class = class_names[pred_idx]

    status = ""
    if true_class is not None:
        status = "Correct" if true_class == pred_class else "Incorrect"

    title_text = f"{model_name}\nTrue: {true_class} | Pred: {pred_class}"
    if status:
        title_text += f" | {status}"
    if severity is not None:
        title_text += f"\nSeverity: {severity}"

    plt.figure(figsize=(10, 4))

    plt.subplot(1, 2, 1)
    plt.imshow(img_uint8.astype(np.uint8))
    plt.title("Original")
    plt.axis("off")

    plt.subplot(1, 2, 2)
    plt.imshow(cv2.cvtColor(superimposed, cv2.COLOR_BGR2RGB))
    plt.title(title_text)
    plt.axis("off")

    plt.tight_layout()
    plt.savefig(out_path, dpi=200, bbox_inches="tight")
    plt.close()


gradcam_dir = os.path.join(PLOTS_DIR, "gradcam_samples")
os.makedirs(gradcam_dir, exist_ok=True)

# Optional clear old Grad-CAM files
for item in os.listdir(gradcam_dir):
    item_path = os.path.join(gradcam_dir, item)
    if os.path.isfile(item_path):
        os.remove(item_path)
    elif os.path.isdir(item_path):
        shutil.rmtree(item_path)

# Select Grad-CAM samples from severity groups only
selected_parts = []
severity_order = ["mild", "moderate", "severe"]
TOTAL_TARGET = 20
PER_SEVERITY_TARGET = max(1, TOTAL_TARGET // len(severity_order))

for sev in severity_order:
    sev_df = severity_df[severity_df["severity"] == sev].copy()
    if len(sev_df) == 0:
        continue

    chosen_df = sev_df.sample(
        n=min(PER_SEVERITY_TARGET, len(sev_df)),
        random_state=SEED
    )
    selected_parts.append(chosen_df)

sample_df = pd.concat(selected_parts, ignore_index=True) if len(selected_parts) > 0 else pd.DataFrame()

if len(sample_df) < TOTAL_TARGET:
    remaining_pool = severity_df[severity_df["severity"].isin(severity_order)].drop_duplicates(subset=["image_path"])
    if len(sample_df) > 0:
        remaining_pool = remaining_pool[~remaining_pool["image_path"].isin(sample_df["image_path"])]

    if len(remaining_pool) > 0:
        extra_n = min(TOTAL_TARGET - len(sample_df), len(remaining_pool))
        extra_df = remaining_pool.sample(n=extra_n, random_state=SEED)
        sample_df = pd.concat([sample_df, extra_df], ignore_index=True)

sample_df = sample_df.head(TOTAL_TARGET).copy()

print(f"\nTotal Grad-CAM base samples selected: {len(sample_df)}")
if len(sample_df) > 0:
    print(sample_df["severity"].value_counts(dropna=False))

gradcam_rows = []

# Keep all models
gradcam_model_list = summary_df["Model"].tolist()

for _, mrow in summary_df.iterrows():
    model_name = mrow["Model"]
    if model_name not in gradcam_model_list:
        continue

    model_path = mrow["Model_Path"]
    print(f"\nGenerating Grad-CAM for: {model_name}")
    model = load_saved_model(model_path)

    model_gradcam_dir = os.path.join(gradcam_dir, model_name)
    os.makedirs(model_gradcam_dir, exist_ok=True)

    # Batched predictions first for chosen sample set
    sample_image_paths = sample_df["image_path"].tolist()
    pred_idx_arr, _ = batched_predict_from_paths(
        model, model_name, sample_image_paths, batch_size=BATCH_SIZE
    )

    model_pred_rows = []
    for idx_row, (_, row) in enumerate(sample_df.iterrows()):
        pred_idx = int(pred_idx_arr[idx_row])
        pred_class = class_names[pred_idx]
        is_correct = int(pred_idx == int(row["true_idx"]))

        model_pred_rows.append({
            "image_path": row["image_path"],
            "true_class": row["true_class"],
            "true_idx": int(row["true_idx"]),
            "pred_idx": pred_idx,
            "pred_class": pred_class,
            "severity": row["severity"],
            "correct": is_correct
        })

    model_pred_df = pd.DataFrame(model_pred_rows)
    model_pred_df.to_csv(
        os.path.join(CSV_DIR, f"{model_name}_gradcam_sample_predictions.csv"),
        index=False
    )

    for i, prow in model_pred_df.iterrows():
        img_path = prow["image_path"]
        fname = os.path.basename(img_path)
        correctness_tag = "correct" if int(prow["correct"]) == 1 else "wrong"

        out_path = os.path.join(
            model_gradcam_dir,
            f"{model_name}_{prow['severity']}_{correctness_tag}_{i}_{fname}.png"
        )

        save_gradcam_overlay(
            img_path=img_path,
            model=model,
            model_name=model_name,
            out_path=out_path,
            true_class=prow["true_class"],
            severity=prow["severity"]
        )

        gradcam_rows.append({
            "model_name": model_name,
            "image_path": img_path,
            "output_path": out_path,
            "true_class": prow["true_class"],
            "pred_class": prow["pred_class"],
            "severity": prow["severity"],
            "correct": int(prow["correct"])
        })

gradcam_df = pd.DataFrame(gradcam_rows)
gradcam_df.to_csv(os.path.join(CSV_DIR, "experiment_group4_gradcam_files.csv"), index=False)

print("\nSaved Grad-CAM file summary:")
if len(gradcam_df) > 0:
    print(gradcam_df[["model_name", "true_class", "pred_class", "severity", "correct", "output_path"]].head(50))
else:
    print("No Grad-CAM files saved.")

# =========================
# EXPERIMENT GROUP 5
# EFFICIENCY
# =========================
def measure_inference_time(model, model_name, sample_paths, warmup_batches=2, batch_size=BATCH_SIZE):
    if len(sample_paths) == 0:
        return np.nan

    ds = preprocess_batch_image_paths(sample_paths, model_name, batch_size=batch_size)

    # Warmup
    for i, batch in enumerate(ds.take(warmup_batches)):
        _ = model.predict(batch, verbose=0)

    # Actual timing
    n_images = 0
    start = time.time()
    for batch in ds:
        preds = model.predict(batch, verbose=0)
        n_images += preds.shape[0]
    end = time.time()

    return (end - start) / max(n_images, 1)


eff_rows = []

# Use subset for fair timing if test set is large
timing_paths = test_index_df["image_path"].tolist()
timing_paths = timing_paths[:min(len(timing_paths), 256)]

for _, row in summary_df.iterrows():
    model_name = row["Model"]
    model_path = row["Model_Path"]
    model = load_saved_model(model_path)

    model_size_mb = os.path.getsize(model_path) / (1024 * 1024)
    avg_inf_time = measure_inference_time(
        model,
        model_name,
        timing_paths,
        warmup_batches=2,
        batch_size=BATCH_SIZE
    )

    eff_rows.append({
        "Model": model_name,
        "Parameters_M": float(model.count_params() / 1e6),
        "Model_Size_MB": float(model_size_mb),
        "Avg_Inference_Time_sec_per_image": float(avg_inf_time)
    })

eff_df = pd.DataFrame(eff_rows).sort_values("Avg_Inference_Time_sec_per_image")
eff_df.to_csv(os.path.join(CSV_DIR, "experiment_group5_efficiency.csv"), index=False)

print("\n=== Efficiency ===")
print(eff_df)

# =========================
# FINAL MERGE
# =========================
final_df = summary_df.merge(eff_df, on="Model", how="left")
final_df.to_csv(os.path.join(CSV_DIR, "final_experiment_summary.csv"), index=False)

print("\n=== Final Summary ===")
print(final_df)
print("\nAll important files saved in:", RESULTS_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
TensorFlow version: 2.19.0
Available GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
GPU memory growth enabled.

=== NVIDIA GPU INFO ===
Mon Mar 23 18:53:13 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                  

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()



=== Severity Results ===
   Severity  Count  Accuracy  Macro_F1
0      mild    569  0.985940  0.743652
1  moderate    299  0.996656  0.748869
2    severe    221  0.959276  0.734274

Total Grad-CAM base samples selected: 20
severity
mild        7
moderate    7
severe      6
Name: count, dtype: int64

Generating Grad-CAM for: ResNet50_baseline


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()
/usr/local/lib/python3.12/dist-packages/keras/src/models/functional.py:241: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: [['input_layer_1']]
Received: inputs=Tensor(shape=(1, 224, 224, 3))
  warnings.warn(msg)



Generating Grad-CAM for: EfficientNetB0_baseline


...


Generating Grad-CAM for: MobileNetV3Small_SE


/usr/local/lib/python3.12/dist-packages/keras/src/models/functional.py:241: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: [['input_layer_2']]
Received: inputs=Tensor(shape=(1, 224, 224, 3))
  warnings.warn(msg)



Generating Grad-CAM for: MobileNetV3Small_CBAM


/usr/local/lib/python3.12/dist-packages/keras/src/models/functional.py:241: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: [['input_layer_3']]
Received: inputs=Tensor(shape=(1, 224, 224, 3))
  warnings.warn(msg)



Generating Grad-CAM for: MobileNetV3Small_baseline

Saved Grad-CAM file summary:
                 model_name             true_class             pred_class  \
0         ResNet50_baseline     Tomato_Late_blight     Tomato_Late_blight   
1         ResNet50_baseline         Bacterial_spot         Bacterial_spot   
2         ResNet50_baseline         Bacterial_spot         Bacterial_spot   
3         ResNet50_baseline  YellowLeaf_Curl_Virus  YellowLeaf_Curl_Virus   
4         ResNet50_baseline  YellowLeaf_Curl_Virus  YellowLeaf_Curl_Virus   
5         ResNet50_baseline  YellowLeaf_Curl_Virus  YellowLeaf_Curl_Virus   
6         ResNet50_baseline  YellowLeaf_Curl_Virus  YellowLeaf_Curl_Virus   
7         ResNet50_baseline  YellowLeaf_Curl_Virus  YellowLeaf_Curl_Virus   
8         ResNet50_baseline     Tomato_Late_blight     Tomato_Late_blight   
9         ResNet50_baseline         Bacterial_spot         Bacterial_spot   
10        ResNet50_baseline         Bacterial_spot         Bacterial_sp